# Blockchain Forensics and Cryptocurrency Tracing

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
import networkx as nx
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
import hashlib
from collections import defaultdict, deque
import re

class BlockchainForensicsAnalyzer:
    def __init__(self):
        self.transaction_graph = nx.DiGraph()
        self.clustering_model = DBSCAN(eps=0.3, min_samples=5)
        self.scaler = StandardScaler()
        self.suspicious_patterns = defaultdict(float)

    def analyze_transaction_flow(self, transactions):
        for tx in transactions:
            self.transaction_graph.add_edge(tx['from'], tx['to'], amount=tx['amount'], timestamp=tx['timestamp'])
            features = [tx['amount'], tx['gas_used'], len(tx['inputs']), len(tx['outputs'])]
            tx_hash = hashlib.sha256(f"{tx['from']}{tx['to']}{tx['amount']}".encode()).hexdigest()
            self.suspicious_patterns[tx_hash] = self.calculate_suspicion_score(tx)
        return self.detect_money_laundering_patterns()

    def calculate_suspicion_score(self, tx):
        score = 0
        if tx['amount'] > 1000000: score += 0.3
        if len(tx['inputs']) > 10: score += 0.2
        if tx['timestamp'] % 86400 < 3600 or tx['timestamp'] % 86400 > 82800: score += 0.1
        mixer_patterns = ['mix', 'tumbl', 'priv', 'anon']
        if any(pattern in tx['memo'].lower() for pattern in mixer_patterns): score += 0.4
        return score

    def detect_money_laundering_patterns(self):
        suspicious_clusters = []
        for component in nx.strongly_connected_components(self.transaction_graph):
            if len(component) > 3:
                subgraph = self.transaction_graph.subgraph(component)
                total_volume = sum(data['amount'] for _, _, data in subgraph.edges(data=True))
                avg_degree = np.mean([subgraph.degree(node) for node in component])
                if total_volume > 500000 and avg_degree > 5:
                    suspicious_clusters.append({'addresses': component, 'volume': total_volume})
        return suspicious_clusters

    def trace_fund_origins(self, target_address, max_depth=6):
        origins = []
        visited = set()
        queue = deque([(target_address, 0)])
        while queue:
            addr, depth = queue.popleft()
            if depth >= max_depth or addr in visited: continue
            visited.add(addr)
            predecessors = list(self.transaction_graph.predecessors(addr))
            if not predecessors:
                origins.append({'origin': addr, 'depth': depth})
            else:
                queue.extend([(pred, depth + 1) for pred in predecessors])
        return origins

    def identify_mixing_services(self, transactions):
        mixing_indicators = []
        address_activity = defaultdict(lambda: {'in': 0, 'out': 0, 'unique_counterparts': set()})
        for tx in transactions:
            address_activity[tx['to']]['in'] += tx['amount']
            address_activity[tx['to']]['unique_counterparts'].add(tx['from'])
            address_activity[tx['from']]['out'] += tx['amount']
            address_activity[tx['from']]['unique_counterparts'].add(tx['to'])

        for addr, activity in address_activity.items():
            ratio = activity['in'] / max(activity['out'], 1)
            unique_count = len(activity['unique_counterparts'])
            if 0.95 < ratio < 1.05 and unique_count > 50:
                mixing_indicators.append({'address': addr, 'confidence': min(unique_count/100, 1.0)})
        return mixing_indicators